In [7]:
import pandas as pd
import numpy as np

In [3]:
import os
print("Робоча папка Jupyter:", os.getcwd())
print("Файли, які він бачить зараз:", os.listdir())

Робоча папка Jupyter: C:\Users\Home pc\anaconda_projects\37faddfa-ee4c-4da6-bae2-d0cb983bbab4
Файли, які він бачить зараз: ['.ipynb_checkpoints', 'course_project.ipynb']


In [8]:
df = pd.read_csv('bq-results.csv')

In [21]:
df.dtypes

date                object
event_time          object
event_name          object
user_session_id    float64
device_category     object
device_language     object
device_os           object
source              object
medium              object
campaign            object
landing_page        object
dtype: object

In [23]:
df['date'] = pd.to_datetime(df['date'])


In [28]:
df['event_time'] = df['event_time'].str.split('.').str[0]
df['event_time'] = df['event_time'].str.replace(' UTC', '')

In [29]:
df['event_time'] = pd.to_datetime(df['event_time'])


,date,event_time,event_name,user_session_id,device_category,device_language,device_os,source,medium,campaign,landing_page
0,2021-01-02,2021-01-02 12:09:21,session_start,1.019469e+06,mobile,en-us,Web,undefined,(data deleted),(data deleted),https://shop.googlemerchandisestore.com/Google...
1,2021-01-02,2021-01-02 05:38:21,session_start,1.140798e+06,mobile,en,iOS,undefined,(data deleted),(data deleted),https://googlemerchandisestore.com/
2,2021-01-02,2021-01-02 08:49:43,session_start,1.990491e+06,desktop,en-us,Windows,undefined,(data deleted),(data deleted),https://shop.googlemerchandisestore.com/Google...
3,2021-01-02,2021-01-02 09:03:21,session_start,2.507096e+06,desktop,NaN,Web,undefined,(data deleted),(data deleted),https://shop.googlemerchandisestore.com/Google...
4,2021-01-02,2021-01-02 12:31:07,session_start,2.537785e+06,mobile,es-es,Web,undefined,(data deleted),(data deleted),https://shop.googlemerchandisestore.com/Google...


In [30]:
df.dtypes

date               datetime64[ns]
event_time         datetime64[ns]
event_name                 object
user_session_id           float64
device_category            object
device_language            object
device_os                  object
source                     object
medium                     object
campaign                   object
landing_page               object
dtype: object

In [31]:
##перевіряємо пусті значення
nan_counts = df.isna().sum()
print(nan_counts)

date                    0
event_time              0
event_name              0
user_session_id         0
device_category         0
device_language    373381
device_os               0
source                  0
medium                  0
campaign                0
landing_page            0
dtype: int64


In [34]:
##заповнюємо пусті значення
df['device_language'] = df['device_language'].fillna('undefined')


In [35]:
df['device_language'].isna().sum()

np.int64(0)

In [38]:
##створюємо список для того щоб автоматизувати перевірку унікальних значень в ствобцях
target_cols = [
    'device_category', 'device_language', 'device_os', 
    'source', 'medium', 'campaign', 'landing_page', 'event_name'
]

In [39]:
##бачимо,що є сирі дані, які потрібно почистити, наприклад data deleted
for col in target_cols:
    print(f"--- {col} ({df[col].nunique()} унікальних значень) ---")
    print(df[col].unique())
    print("-" * 30) # Гарна лінія розділення

--- device_category (3 унікальних значень) ---
['mobile' 'desktop' 'tablet']
------------------------------
--- device_language (10 унікальних значень) ---
['en-us' 'en' 'undefined' 'es-es' 'de' 'en-gb' 'ko' 'zh' 'en-ca' 'fr']
------------------------------
--- device_os (6 унікальних значень) ---
['Web' 'iOS' 'Windows' 'Macintosh' 'Android' '<Other>']
------------------------------
--- source (5 унікальних значень) ---
['undefined' '<Other>' '(direct)' 'google'
 'shop.googlemerchandisestore.com']
------------------------------
--- medium (6 унікальних значень) ---
['(data deleted)' '(none)' '<Other>' 'cpc' 'organic' 'referral']
------------------------------
--- campaign (5 унікальних значень) ---
['(data deleted)' '<Other>' '(direct)' '(organic)' '(referral)']
------------------------------
--- landing_page (1331 унікальних значень) ---
['https://shop.googlemerchandisestore.com/Google+Redesign/Apparel/YouTube+Icon+Hoodie+Black'
 'https://googlemerchandisestore.com/'
 'https://shop.go

In [17]:
#змінюємо спочатку один стовбець
print(df['source'].value_counts())

source
google                             300333
<Other>                            227244
(direct)                           200992
shop.googlemerchandisestore.com     78687
(data deleted)                      70395
Name: count, dtype: int64


In [51]:
df['source'] = df['source'].replace({'(data deleted)': 'undefined','<Other>': 'Other'})

In [52]:
print(df['source'].value_counts())

source
google                             300333
Other                              227244
(direct)                           200992
shop.googlemerchandisestore.com     78687
undefined                           70395
Name: count, dtype: int64


In [41]:
#зараз створюємо список з стовбцями та змінемо в них значення.
columns_to_fix = ['medium', 'campaign']

In [43]:
#змінюємо дані в стовбцях
df[columns_to_fix] = df[columns_to_fix].replace('(data deleted)', 'undefined')

In [53]:
for col in columns_to_fix:
    print(f"--- {col} ({df[col].nunique()} унікальних значень) ---")
    print(df[col].unique())
    print("-" * 30) # Гарна лінія розділення

--- medium (5 унікальних значень) ---
['undefined' 'Other' 'cpc' 'organic' 'referral']
------------------------------
--- campaign (5 унікальних значень) ---
['undefined' '<Other>' '(direct)' '(organic)' '(referral)']
------------------------------


In [54]:
#змінюємо '(none)' та '<Other>'
df['medium'] = df['medium'].replace({'(none)': 'undefined', '<Other>': 'Other'})


In [50]:
df['medium'].value_counts()

medium
organic      288682
undefined    272498
referral     162874
Other        118833
cpc           34764
Name: count, dtype: int64

In [55]:
#змінюємо значення з <>() 
df['campaign'] = df['campaign'].str.replace(r'[<>()]', '', regex=True).str.strip()

In [56]:
df['medium'].value_counts()

medium
organic      288682
undefined    272498
referral     162874
Other        118833
cpc           34764
Name: count, dtype: int64

In [57]:
#змінюємо значення з <> 
df['device_os'] = df['device_os'].replace('<Other>', 'Other')

In [58]:
df.head()

,date,event_time,event_name,user_session_id,device_category,device_language,device_os,source,medium,campaign,landing_page
0,2021-01-02,2021-01-02 12:09:21,session_start,1.019469e+06,mobile,en-us,Web,undefined,undefined,undefined,https://shop.googlemerchandisestore.com/Google...
1,2021-01-02,2021-01-02 05:38:21,session_start,1.140798e+06,mobile,en,iOS,undefined,undefined,undefined,https://googlemerchandisestore.com/
2,2021-01-02,2021-01-02 08:49:43,session_start,1.990491e+06,desktop,en-us,Windows,undefined,undefined,undefined,https://shop.googlemerchandisestore.com/Google...
3,2021-01-02,2021-01-02 09:03:21,session_start,2.507096e+06,desktop,undefined,Web,undefined,undefined,undefined,https://shop.googlemerchandisestore.com/Google...
4,2021-01-02,2021-01-02 12:31:07,session_start,2.537785e+06,mobile,es-es,Web,undefined,undefined,undefined,https://shop.googlemerchandisestore.com/Google...


In [59]:
df.to_csv('cleaned_ecommerce_data.csv', index=False)